## Importing Libraries

In [1]:
import os
import re
import shutil
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from datetime import date
from IPython.display import display

tqdm.pandas()

In [2]:
processed_file_path = 'Data/processed'
os.makedirs(processed_file_path, exist_ok = True)

## Data Pre-Processing

### 1. Weather Data

**Column Details**

- `District` : Name of the district
- `Date` : Date of record
- `RainFall` : Average Cumulative Rainfall in mm
- `Min_Temp` : Average Minimum Temperature in celcius
- `Max_Temp` : Average Maximum Temperature in celcius
- `Min_Humidity` : AverageMinimum Humidity %
- `Max_Humidity` : Average Maximum Humidity %

In [3]:
Weather_DF = pd.read_parquet('Data/interim/Weather_Data.parquet', engine = 'pyarrow')

print(f'Number of Districts in Weather Dataset\t\t: {Weather_DF.District.nunique()}')
print(f'Number of data points in the  Weather Dataset\t: {Weather_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t: {sum(Weather_DF.isnull().sum())}\n')

Weather_DF.head()

Number of Districts in Weather Dataset		: 33
Number of data points in the  Weather Dataset	: 1356051
Number of missing values in the dataset		: 0



,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Adilabad,2019-01-01,0.0,6.2,27.6,17.2,88.0
1,Adilabad,2019-01-01,0.0,6.0,26.6,21.6,96.4
2,Adilabad,2019-01-01,0.0,5.7,27.4,18.8,75.1
3,Adilabad,2019-01-01,0.0,8.1,28.1,20.5,64.0
4,Adilabad,2019-01-01,0.0,9.5,25.6,20.0,66.0


In [4]:
print('>>> Descriptive statistics\n')
display(Weather_DF.describe())

print('\n>>> Information about Weather dataFrame\n')
Weather_DF.info()

>>> Descriptive statistics



,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
count,1356051,1.356051e+06,1.356051e+06,1.356051e+06,1.356051e+06,1.356051e+06
mean,2021-11-23 14:37:29.644592896,3.049499e+00,2.245977e+01,3.348291e+01,4.717251e+01,8.836494e+01
min,2019-01-01 00:00:00,0.000000e+00,0.000000e+00,3.600000e+00,0.000000e+00,0.000000e+00
25%,2020-04-27 00:00:00,0.000000e+00,1.970000e+01,3.090000e+01,3.070000e+01,8.290000e+01
50%,2021-11-06 00:00:00,0.000000e+00,2.310000e+01,3.320000e+01,4.600000e+01,9.440000e+01
75%,2023-07-01 00:00:00,0.000000e+00,2.510000e+01,3.670000e+01,6.280000e+01,9.990000e+01
max,2024-12-31 00:00:00,6.185000e+02,3.740000e+01,4.790000e+01,1.000000e+02,1.000000e+02
std,NaN,1.123218e+01,4.546990e+00,5.018258e+00,2.102472e+01,1.560613e+01



>>> Information about Weather dataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1356051 entries, 0 to 1356050
Data columns (total 7 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   District      1356051 non-null  string        
 1   Date          1356051 non-null  datetime64[ns]
 2   RainFall      1356051 non-null  float64       
 3   Min_Temp      1356051 non-null  float64       
 4   Max_Temp      1356051 non-null  float64       
 5   Min_Humidity  1356051 non-null  float64       
 6   Max_Humidity  1356051 non-null  float64       
dtypes: datetime64[ns](1), float64(5), string(1)
memory usage: 72.4 MB


#### 1.1 Data Groupping

Now we have day-to-day values, for better comparision and analysis, instead of day-to-day values, converting into Month wise data.

In [5]:
Weather_DF['Year'] = Weather_DF.Date.dt.year
Weather_DF['Month'] = Weather_DF.Date.dt.month

Weather_Grouped = Weather_DF.drop('Date', axis = 1).groupby(['District', 'Year', 'Month']) \
                                                   .agg('mean').reset_index().copy(deep = True)

**NOTE :**

Creating new `Date` column, `Day` will be 01 for all (it's monthly average values, so taking *Day* as 01), Month and Year will come from `Weather_Grouped` dataframes `Month` and `Year` columns respectively.

In [6]:
Weather_Grouped['Month'] = pd.to_datetime(Weather_Grouped.apply(lambda row :
                                    f"01/{row.Month}/{row.Year}", axis = 1), format = '%d/%m/%Y')

In [7]:
# Removing unwanted columns
Weather_Grouped.drop('Year', axis = 1, inplace = True)

# Ordering column names
col_order = ['Month', 'District', 'RainFall', 'Min_Temp',
             'Max_Temp', 'Min_Humidity', 'Max_Humidity']
Weather_Grouped = Weather_Grouped[col_order]

# Sorting dataframe
Weather_Grouped = Weather_Grouped.sort_values(['Month', 'District']).reset_index(drop = True)

original_size = Weather_DF.shape[0]
grouped_size = Weather_Grouped.shape[0]
missingCount = sum(Weather_Grouped.isnull().sum())
print(f'Number of data points in the initial Weather Dataset\t: {original_size}')
print(f'Number of data points in the grouped Weather Dataset\t: {grouped_size}')
print(f'Number of missing values in the grouped Weather dataset\t: {missingCount}\n')

Weather_Grouped.head()

Number of data points in the initial Weather Dataset	: 1356051
Number of data points in the grouped Weather Dataset	: 2333
Number of missing values in the grouped Weather dataset	: 0



,Month,District,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,2019-01-01,Adilabad,0.130856,12.201262,29.281907,33.557363,83.543058
1,2019-01-01,Bhadradri Kothagudem,0.399778,16.166407,30.902670,45.012013,96.371079
2,2019-01-01,Hyderabad,0.942028,17.713180,30.756221,34.638618,81.040737
3,2019-01-01,Jagtial,0.735484,14.104624,29.920323,37.005914,92.996452
4,2019-01-01,Jangaon,0.837742,15.170161,29.263065,38.379839,89.622903


In [8]:
print('>>> Descriptive statistics of grouped Weather Data\n')
Weather_Grouped.describe()

>>> Descriptive statistics of grouped Weather Data



,Month,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
count,2333,2333.000000,2333.000000,2333.000000,2333.000000,2333.000000
mean,2021-12-28 17:04:36.210887424,3.155396,22.354275,33.321535,48.007126,89.341225
min,2019-01-01 00:00:00,0.000000,11.214286,13.190538,13.310714,36.029874
25%,2020-07-01 00:00:00,0.091538,19.314259,31.205279,34.449167,85.263930
50%,2022-01-01 00:00:00,0.967494,23.155054,32.766244,46.641944,93.773684
75%,2023-07-01 00:00:00,5.113978,24.948148,36.489259,63.419916,96.573041
max,2024-12-01 00:00:00,33.508602,33.856802,43.749247,83.868871,100.000000
std,NaN,4.486925,4.089226,4.389334,17.073126,10.605450


#### 1.2 Saving Processed Weather Dataset

In [9]:
# Saving DataFrame to Parquet

weather_path = f'{processed_file_path}/Weather_Data_Processed.csv'
print(f'Saving processed Weather Dataset to "{weather_path}"')
if not os.path.isfile(weather_path):
    Weather_Grouped.to_csv(weather_path, index = False)

Saving processed Weather Dataset to "Data/processed/Weather_Data_Processed.csv"


### 2. Registration and Stamps Data (Non-Agriculture)

**Column Details**

- `Date` : Data of documents registered
- `Dist_Name` : District Name
- `Documents_Registered_Cnt` : Total Documents Registered Count
- `Documents_Registered_Rev` : Total Documents Registered Revenue
- `Estamps_Challans_Cnt` : E-stamps challans count
- `Estamps_Challans_Rev` : E-stamps challans revenue
- `Slot_Booking_Cnt` : Total online Slot Booking Count

In [10]:
registration_DF = pd.read_parquet('Data/interim/Registration_Data.parquet', engine = 'pyarrow')

print(f'Number of Districts in Registration Dataset\t\t: {registration_DF.Dist_Name.nunique()}')
print(f'Number of data points in the  Registration Dataset\t: {registration_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t\t: {sum(registration_DF.isnull().sum())}\n')

registration_DF.head()

Number of Districts in Registration Dataset		: 33
Number of data points in the  Registration Dataset	: 306677
Number of missing values in the dataset			: 0



,Date,Dist_Name,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
0,2019-01-01,Bhadradri Kothagudem,0,0,0,0,0
1,2019-01-01,Ranga Reddy,0,0,0,0,1
2,2019-01-01,Siddipet,0,0,0,0,7
3,2019-01-01,Rajanna Sircilla,0,0,0,0,0
4,2019-01-01,Karimnagar,0,0,0,0,0


In [11]:
print('>>> Descriptive statistics\n')
display(registration_DF.describe())

print('\n>>> Information about Registration dataFrame\n')
registration_DF.info()

>>> Descriptive statistics



,Date,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
count,306677,306677.000000,3.066770e+05,306677.000000,3.066770e+05,306677.000000
mean,2022-01-06 16:40:31.514590464,23.978104,1.655724e+06,16.490112,1.364361e+06,2.054060
min,2019-01-01 00:00:00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,2020-06-27 00:00:00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
50%,2022-01-22 00:00:00,13.000000,2.152500e+05,2.000000,4.020000e+04,0.000000
75%,2023-07-18 00:00:00,36.000000,1.157000e+06,24.000000,7.962400e+05,0.000000
max,2024-12-31 00:00:00,1235.000000,5.351640e+08,549.000000,5.198990e+08,473.000000
std,NaN,33.223862,6.270801e+06,27.222562,5.690234e+06,8.780325



>>> Information about Registration dataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306677 entries, 0 to 306676
Data columns (total 7 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Date                      306677 non-null  datetime64[ns]
 1   Dist_Name                 306677 non-null  string        
 2   Documents_Registered_Cnt  306677 non-null  int64         
 3   Documents_Registered_Rev  306677 non-null  int64         
 4   Estamps_Challans_Cnt      306677 non-null  int64         
 5   Estamps_Challans_Rev      306677 non-null  int64         
 6   Slot_Booking_Cnt          306677 non-null  int64         
dtypes: datetime64[ns](1), int64(5), string(1)
memory usage: 16.4 MB


#### 2.1 Data Groupping

Now we have day-to-day values, for better comparision and analysis, instead of day-to-day values, converting into Month wise data.

In [12]:
registration_DF['Year'] = registration_DF.Date.dt.year
registration_DF['Month'] = registration_DF.Date.dt.month

registration_Grouped = registration_DF.drop('Date', axis = 1).groupby(['Dist_Name',
                                    'Year', 'Month']) .agg('sum').reset_index().copy(deep = True)

**NOTE :**

Creating new `Date` column, `Day` will be 01 for all (it's monthly total count,ie summation, so taking *Day* as 01), Month and Year will come from `registration_Grouped` dataframes `Month` and `Year` columns respectively.

In [13]:
registration_Grouped['Month'] = pd.to_datetime(registration_Grouped.apply(lambda row :
                                    f"01/{row.Month}/{row.Year}", axis = 1), format = '%d/%m/%Y')

In [14]:
# Removing unwanted columns
registration_Grouped.drop('Year', axis = 1, inplace = True)

# Ordering column names
col_order = ['Month', 'Dist_Name', 'Documents_Registered_Cnt', 'Documents_Registered_Rev',
             'Estamps_Challans_Cnt', 'Estamps_Challans_Rev', 'Slot_Booking_Cnt']
registration_Grouped = registration_Grouped[col_order]

# Sorting dataframe
registration_Grouped = registration_Grouped.sort_values(
                                            ['Month', 'Dist_Name']).reset_index(drop = True)

original_size = registration_DF.shape[0]
grouped_size = registration_Grouped.shape[0]
missingCount = sum(registration_Grouped.isnull().sum())
print(f'Number of data points in the initial Registration Dataset\t: {original_size}')
print(f'Number of data points in the grouped Registration Dataset\t: {grouped_size}')
print(f'Number of missing values in the grouped Registration dataset\t: {missingCount}\n')

registration_Grouped.head()

Number of data points in the initial Registration Dataset	: 306677
Number of data points in the grouped Registration Dataset	: 2290
Number of missing values in the grouped Registration dataset	: 0



,Month,Dist_Name,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
0,2019-01-01,Adilabad,945,12390300,0,0,877
1,2019-01-01,Bhadradri Kothagudem,557,9981040,0,0,38
2,2019-01-01,Hanumakonda,3913,120275730,0,0,859
3,2019-01-01,Hyderabad,5206,643949380,0,0,236
4,2019-01-01,Jagtial,2181,28766596,0,0,904


In [15]:
print('>>> Descriptive statistics of grouped Registration Data\n')
registration_Grouped.describe()

>>> Descriptive statistics of grouped Registration Data



,Month,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
count,2290,2290.000000,2.290000e+03,2290.000000,2.290000e+03,2290.000000
mean,2021-12-23 21:14:37.205240064,3211.149782,2.217346e+08,2208.357205,1.827154e+08,275.079913
min,2019-01-01 00:00:00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,2020-06-01 00:00:00,1091.250000,2.112702e+07,0.000000,0.000000e+00,0.000000
50%,2022-01-01 00:00:00,1897.000000,3.866774e+07,1205.000000,2.725687e+07,3.000000
75%,2023-07-01 00:00:00,3836.500000,1.014668e+08,2625.750000,7.301116e+07,212.750000
max,2024-12-01 00:00:00,33786.000000,5.068863e+09,30526.000000,5.239561e+09,11575.000000
std,NaN,4135.171127,5.874795e+08,3775.365279,5.715230e+08,627.762648


#### 2.2 Saving Processed Registration Dataset

In [16]:
# Saving DataFrame to Parquet

registration_path = f'{processed_file_path}/Registration_Data_Processed.csv'
print(f'Saving processed Registration Dataset to "{registration_path}"')
if not os.path.isfile(registration_path):
    registration_Grouped.to_csv(registration_path, index = False)

Saving processed Registration Dataset to "Data/processed/Registration_Data_Processed.csv"


### 3. Telangana Industries TS-iPASS Data

**Column Details**

- `District_Name` : Names of district
- `Name_Of_The_Unit` : Name and address of the unit
- `Line_Of_Activity` : Line of activiity of the organization
- `Sector` : Sector which the industry belongs to
- `Investment` : Investment amount in Millions
- `Number_Of_Employees` : Number of employees in the organization
- `Application_Date` : Date of the application registered in the government for approval
- `Approval_Date` : Date of approval for the application submitted
- `Progress_Of_Implementation` : Indicated status of the businesses
- `Social_Status` : Category of the applicant

In [17]:
ipass_DF = pd.read_parquet('Data/interim/TSiPASS_Data.parquet', engine = 'pyarrow')

original_size = ipass_DF.shape
missingCount = sum(ipass_DF.isnull().sum())
print(f'Number of Districts in TS-iPASS Dataset\t\t: {ipass_DF.District_Name.nunique()}')
print(f'Number of data points in the TS-iPASS Dataset\t: {original_size[0]}')
print(f'Number of missing values in the dataset\t\t: {missingCount}\n')

ipass_DF.head()

Number of Districts in TS-iPASS Dataset		: 33
Number of data points in the TS-iPASS Dataset	: 18000
Number of missing values in the dataset		: 0



,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
0,Medchal-Malkajgiri,Navateja Marketing Pvt Ltd,Manufacturing of iodized salt fromcrude/ raw salt,Food Processing,0.1200,10,2019-01-01,2019-07-19,Commenced Operations,General
1,Ranga Reddy,Sri Koteswara Cam Systems Pvt Ltd,Engineering and fabrication units (dry process...,Engineering,9.1100,125,2019-01-01,2019-04-17,Commenced Operations,General
2,Sangareddy,Sanjay Technical Services Private Ltd,Engineering and fabrication units (dry process...,Engineering,5.0550,40,2019-01-01,2019-05-02,Advanced Stage,General
3,Jagtial,Nishan Singh Engineering Works,Engineering and fabrication units (dry process...,Engineering,0.0000,2,2019-01-02,2019-01-16,Commenced Operations,OBC
4,Jangaon,Dew Industries,Engineering and fabrication units (dry process...,Engineering,0.4916,8,2019-01-02,2019-01-11,Commenced Operations,General


In [18]:
print('>>> Descriptive statistics\n')
display(ipass_DF.describe())

print('\n>>> Information about TS-iPASS dataFrame\n')
ipass_DF.info()

>>> Descriptive statistics



,Investment,Number_Of_Employees,Application_Date,Approval_Date
count,18000.000000,18000.000000,18000,18000
mean,7.043268,39.501944,2021-10-04 01:39:07.200000256,2021-11-11 01:05:45.600000
min,0.000000,0.000000,2019-01-01 00:00:00,2019-01-04 00:00:00
25%,0.150000,5.000000,2020-08-04 00:00:00,2020-09-16 00:00:00
50%,0.250000,9.000000,2021-09-13 00:00:00,2021-10-23 00:00:00
75%,1.000000,15.000000,2022-12-14 00:00:00,2023-01-31 00:00:00
max,9528.000000,40541.000000,2024-12-31 00:00:00,2025-01-24 00:00:00
std,115.912144,636.854634,NaN,NaN



>>> Information about TS-iPASS dataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   District_Name               18000 non-null  string        
 1   Name_Of_The_Unit            18000 non-null  string        
 2   Line_Of_Activity            18000 non-null  string        
 3   Sector                      18000 non-null  string        
 4   Investment                  18000 non-null  float64       
 5   Number_Of_Employees         18000 non-null  int64         
 6   Application_Date            18000 non-null  datetime64[ns]
 7   Approval_Date               18000 non-null  datetime64[ns]
 8   Progress_Of_Implementation  18000 non-null  string        
 9   Social_Status               18000 non-null  string        
dtypes: datetime64[ns](2), float64(1), int64(1), string(6)
memory usage: 1.4 MB


#### 3.1 Data Filtering

- Till now we considered, project application from `January, 2019` to `December, 2024`.
- But there are projects that applied in `December, 2024` or earlier, but got approved in year `2025`.
- So we are filtering the data once again using `Approval_Date` before the analysis.

In [19]:
ipass_DF = ipass_DF.query('Approval_Date < 2025') \
                   .sort_values('Approval_Date').reset_index(drop = True)

filtered_size = ipass_DF.shape

print(f'Number of data points in orginal TS-iPASS Dataset  : {original_size[0]}')
print(f'Number of data points in filtered TS-iPASS Dataset : {filtered_size[0]}\n')

print('>>> First 3 rows of dataset')
display(ipass_DF.head(3))
print('\n>>> Last 3 rows of dataset')
ipass_DF.tail(3)

Number of data points in orginal TS-iPASS Dataset  : 18000
Number of data points in filtered TS-iPASS Dataset : 17949

>>> First 3 rows of dataset


,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
0,Mancherial,M/S R.S. Steels,Steel and steel products using various furnace...,Engineering,1.0338,20,2019-01-03,2019-01-04,Commenced Operations,OBC
1,Kamareddy,Borrolla Pocha Goud,Flour mills (dry process),Food Processing,0.0400,5,2019-01-02,2019-01-07,Commenced Operations,OBC
2,Medchal-Malkajgiri,Kings Mining Tools,Heat treatment with any of the new technology ...,Engineering,0.1100,5,2019-01-07,2019-01-07,Commenced Operations,General



>>> Last 3 rows of dataset


,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
17946,Narayanpet,M/S Om Shakti & Co,Stone crushers,Granite and Stone Crushing,0.7300,15,2024-12-12,2024-12-31,Yet To Start Construction,OBC
17947,Ranga Reddy,Ushakironmovies Pvt Ltd(Thermocol Waste Division),Thermocol manufacturing (with boiler),Others,0.3765,5,2024-12-24,2024-12-31,Yet To Start Construction,General
17948,Medchal-Malkajgiri,M/S. Thaanvika Fabrications And Glass,Engineering and fabrication units (dry process...,Engineering,0.2500,4,2024-12-25,2024-12-31,Yet To Start Construction,General


#### 3.2 Data Encoding

In [20]:
ipass_DF.Social_Status = ipass_DF.Social_Status.apply(lambda x : re.sub(r'SC|ST', 'SC_ST', x))

In [21]:
social_encoded_ = pd.get_dummies(ipass_DF.Social_Status, prefix = 'social_status')

In [22]:
dropCols = ['Name_Of_The_Unit', 'Application_Date', 'Line_Of_Activity',
            'Progress_Of_Implementation', 'Social_Status']
ipass_DF = ipass_DF.drop(dropCols, axis = 1)

ipass_DF = pd.concat([ipass_DF, social_encoded_], axis = 1)

In [23]:
current_size = ipass_DF.shape
missingCount = sum(ipass_DF.isnull().sum())

print(f'Shape of initial TS-iPASS Registration Dataset  : {original_size}')
print(f'Shape of filtered TS-iPASS Registration Dataset : {filtered_size}')
print(f'Shape of present TS-iPASS Registration Dataset  : {current_size}')
print(f'No.of issing values in the present dataset\t: {missingCount}\n')

ipass_DF.head()

Shape of initial TS-iPASS Registration Dataset  : (18000, 10)
Shape of filtered TS-iPASS Registration Dataset : (17949, 10)
Shape of present TS-iPASS Registration Dataset  : (17949, 8)
No.of issing values in the present dataset	: 0



,District_Name,Sector,Investment,Number_Of_Employees,Approval_Date,social_status_General,social_status_OBC,social_status_SC_ST
0,Mancherial,Engineering,1.0338,20,2019-01-04,False,True,False
1,Kamareddy,Food Processing,0.0400,5,2019-01-07,False,True,False
2,Medchal-Malkajgiri,Engineering,0.1100,5,2019-01-07,True,False,False
3,Kamareddy,"Cement, Cement & Concrete Products, Fly Ash Br...",2.4851,15,2019-01-09,True,False,False
4,Medchal-Malkajgiri,Paper and Printing,0.6000,20,2019-01-09,True,False,False


#### 3.2 Data Groupping

Now we have day-to-day values, for better comparision and analysis, instead of day-to-day values, converting into Month wise data.

In [24]:
ipass_DF['Year'] = ipass_DF.Approval_Date.dt.year
ipass_DF['Month'] = ipass_DF.Approval_Date.dt.month

ipass_Grouped = ipass_DF.drop('Approval_Date', axis = 1) \
                        .groupby(['District_Name', 'Sector', 'Year', 'Month']) \
                        .agg('sum').reset_index().copy(deep = True)

**NOTE :**

Creating new `Date` column, `Day` will be 01 for all (it's monthly total count,ie summation, so taking *Day* as 01), Month and Year will come from `ipass_Grouped` dataframes `Month` and `Year` columns respectively.

In [25]:
ipass_Grouped['Month'] = pd.to_datetime(ipass_Grouped.apply(lambda row :
                                f"01/{row.Month}/{row.Year}", axis = 1), format = '%d/%m/%Y')

In [26]:
# Removing unwanted columns
ipass_Grouped.drop('Year', axis = 1, inplace = True)

# Ordering column names
col_order = ['Month', 'District_Name', 'Sector', 'Investment', 'Number_Of_Employees',
             'social_status_General', 'social_status_OBC', 'social_status_SC_ST']
ipass_Grouped = ipass_Grouped[col_order]

# Sorting dataframe
ipass_Grouped = ipass_Grouped.sort_values(
                ['Month', 'District_Name', 'Sector']).reset_index(drop = True)

grouped_size = ipass_Grouped.shape
missingCount = sum(ipass_Grouped.isnull().sum())
print(f'Number of data points in the initial TS-iPASS Dataset\t : {original_size[0]}')
print(f'Number of data points in the filtered TS-iPASS Dataset\t : {filtered_size[0]}')
print(f'Number of data points in the grouped TS-iPASS Dataset\t : {grouped_size[0]}')
print(f'Number of missing values in the grouped TS-iPASS dataset : {missingCount}\n')

ipass_Grouped.head()

Number of data points in the initial TS-iPASS Dataset	 : 18000
Number of data points in the filtered TS-iPASS Dataset	 : 17949
Number of data points in the grouped TS-iPASS Dataset	 : 7944
Number of missing values in the grouped TS-iPASS dataset : 0



,Month,District_Name,Sector,Investment,Number_Of_Employees,social_status_General,social_status_OBC,social_status_SC_ST
0,2019-01-01,Adilabad,"Cement, Cement & Concrete Products, Fly Ash Br...",0.0700,15,0,0,1
1,2019-01-01,Adilabad,Food Processing,0.0183,1,0,2,0
2,2019-01-01,Bhadradri Kothagudem,Agro based incl Cold Storages,0.0800,4,1,0,0
3,2019-01-01,Bhadradri Kothagudem,"Cement, Cement & Concrete Products, Fly Ash Br...",0.1500,8,0,1,0
4,2019-01-01,Jagtial,Agro based incl Cold Storages,0.4110,11,0,4,0


**Check**

Let's figure out the investments that occurred in `Sangareddy` district in `October 2022`.

In [27]:
month_ = date(2022, 10, 1)
ipass_Grouped.query('District_Name == "Sangareddy" and Month == @month_')

,Month,District_Name,Sector,Investment,Number_Of_Employees,social_status_General,social_status_OBC,social_status_SC_ST
5381,2022-10-01,Sangareddy,Agro based incl Cold Storages,0.9500,10,1,0,0
5382,2022-10-01,Sangareddy,Beverages,9.0815,1030,3,0,0
5383,2022-10-01,Sangareddy,Engineering,9.2466,123,2,2,0
5384,2022-10-01,Sangareddy,Food Processing,9.5334,45,3,0,0
5385,2022-10-01,Sangareddy,Granite and Stone Crushing,7.1597,21,3,0,0
5386,2022-10-01,Sangareddy,Others,1.2700,500,1,0,0
5387,2022-10-01,Sangareddy,Pharmaceuticals and Chemicals,105.7800,1101,6,0,0
5388,2022-10-01,Sangareddy,Plastic and Rubber,10.0000,20,1,0,0
5389,2022-10-01,Sangareddy,R&D,1.7945,25,1,0,0


In [28]:
print('>>> Descriptive statistics of grouped TS-iPASS Data\n')
ipass_Grouped.describe()

>>> Descriptive statistics of grouped TS-iPASS Data



,Month,Investment,Number_Of_Employees,social_status_General,social_status_OBC,social_status_SC_ST
count,7944,7944.000000,7944.000000,7944.000000,7944.000000,7944.000000
mean,2021-11-18 15:20:07.250755328,15.956533,89.422583,1.312689,0.781974,0.164778
min,2019-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2020-08-01 00:00:00,0.234900,7.750000,0.000000,0.000000,0.000000
50%,2021-11-01 00:00:00,0.770500,15.000000,1.000000,0.000000,0.000000
75%,2023-04-01 00:00:00,3.150000,40.000000,2.000000,1.000000,0.000000
max,2024-12-01 00:00:00,17793.350800,57000.000000,40.000000,24.000000,5.000000
std,NaN,241.382457,1078.191164,2.040114,1.249065,0.448998


#### 3.2 Saving Processed Registration Dataset

In [29]:
# Saving DataFrame to Parquet

ipass_path = f'{processed_file_path}/TSiPASS_Data_Processed.csv'
ipass_og_path = f'{processed_file_path}/TSiPASS_Data_RAW.parquet'
print(f'Saving processed TS-iPASS Dataset to "{ipass_path}"')
if not os.path.isfile(ipass_path):
    ipass_Grouped.to_csv(ipass_path, index = False)
    # Saving RAW TSiPASS Data in processed because,
    # in order to find time delay in approval, we need the RAW file
    shutil.copy('Data/interim/TSiPASS_Data.parquet', ipass_og_path)

Saving processed TS-iPASS Dataset to "Data/processed/TSiPASS_Data_Processed.csv"


### 4. RTA Vehicle Online Sales Data

**Column Details**

- `District` : Vehicle Registration District name
- `Model_Desc` : Model Description of the vehicle
- `vehicleClass` : Vehicle Classification
- `Fuel` : Fuel type
- `fromdate` : Date of registration of the vehicle
- `TempRegnNo` : Temporary Registration Number
- `Category` : If the vehicle is Transport or Non Transport
- `SecondVehicle` : Is it a second vehicle of the owner
- `Make_Yr` : Vehicle Make Year
- `Manufacturer_Name`: Name of the Automobile Company

In [30]:
rta_DF = None
for file in sorted(glob('Data/interim/RTA_Reg_Data_P*.parquet')):
    temp = pd.read_parquet(file, engine = 'pyarrow')
    rta_DF = pd.concat([rta_DF, temp], join = 'outer')
    
# Sorting values
rta_DF = rta_DF.sort_values(['fromdate', 'District']).reset_index(drop = True)

original_size = rta_DF.shape
print(f'Number of Districts in RTA Registration Dataset\t\t: {rta_DF.District.nunique()}')
print(f'Number of data points in the RTA Registration Dataset\t: {original_size[0]}')
print(f'Number of missing values in the dataset\t\t\t: {sum(rta_DF.isnull().sum())}\n')

rta_DF.head()

Number of Districts in RTA Registration Dataset		: 33
Number of data points in the RTA Registration Dataset	: 9608322
Number of missing values in the dataset			: 0



,District,Model_Desc,vehicleClass,Fuel,SeatingCapacity,fromdate,TempRegnNo,Category,SecondVehicle,makeYear,Manufacturer_Name
0,Adilabad,ACTIVA 5GWEAS&KS&DCBS(CBS)WSEHEETWHEEL BSIV,Motor Cycle,Petrol,2,2019-01-01,TS01LTR7367,Non Transport,N,2018-01-11,HONDA MOTORCYCLE&SCOOTER(I)P L
1,Adilabad,MAHINDRA XUV5OO FWD W11 BSIV,Motor Car,Diesel,7,2019-01-01,TS16XTR5445,Non Transport,Y,2018-01-08,M/S MAHINDRA &MAHINDRA LTD
2,Adilabad,LIVO WK S &S S W F D &R D B W ALLOYWHEELS BSIV,Motor Cycle,Petrol,2,2019-01-01,TS01LTR7329,Non Transport,N,2018-01-06,HONDA MOTORCYCLE&SCOOTER(I)P L
3,Adilabad,"""PASSION PRO""(I3S-SELF-DRUM-CAST) BSIV",Motor Cycle,Petrol,2,2019-01-01,TS01LTR7424,Non Transport,N,2018-01-12,HERO MOTOCORP LTD
4,Adilabad,CB SHINE W F DRUM B K S&E S W CAST WHEELS BSIV,Motor Cycle,Petrol,2,2019-01-01,TS01LTR7374,Non Transport,N,2018-01-11,HONDA MOTORCYCLE&SCOOTER(I)P L


In [31]:
print('>>> Descriptive statistics\n')
display(rta_DF.describe())

print('\n>>> Information about RTA Registration dataFrame\n')
rta_DF.info()

>>> Descriptive statistics



,SeatingCapacity,fromdate,makeYear
count,9.608322e+06,9608322,9608322
mean,2.576014e+00,2021-08-25 06:45:19.346169856,2021-01-10 02:16:20.406609664
min,0.000000e+00,2019-01-01 00:00:00,1978-01-02 00:00:00
25%,2.000000e+00,2020-02-17 00:00:00,2020-01-01 00:00:00
50%,2.000000e+00,2021-07-18 00:00:00,2021-01-04 00:00:00
75%,2.000000e+00,2023-01-20 00:00:00,2022-01-11 00:00:00
max,6.600000e+01,2024-12-31 00:00:00,2025-03-26 00:00:00
std,2.211998e+00,NaN,NaN



>>> Information about RTA Registration dataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9608322 entries, 0 to 9608321
Data columns (total 11 columns):
 #   Column             Dtype         
---  ------             -----         
 0   District           string        
 1   Model_Desc         string        
 2   vehicleClass       string        
 3   Fuel               string        
 4   SeatingCapacity    int64         
 5   fromdate           datetime64[ns]
 6   TempRegnNo         string        
 7   Category           string        
 8   SecondVehicle      string        
 9   makeYear           datetime64[ns]
 10  Manufacturer_Name  string        
dtypes: datetime64[ns](2), int64(1), string(8)
memory usage: 806.4 MB


#### 4.1 Data Correction

In [32]:
vClass_mapping = {'Auto_Rickshaw' : ['Auto Rickshaw', 'Auto Rikckshaw Private', 'Erickshaw'],
                  'Motor_Cycle' : ['Motor Cycle', 'Motor Cycle For Hire',
                                   'Mopeds And Motorised Cycle'],
                  'Agriculture' : ['Tractor For Agricultural Purpose',
                                           'Tractor Driven Combined Harvester',
                                           'Trailer For Agriculture Purpose',
                                           'Self Propelled Harvester'],
                  'Motor_Car' : ['Jeep', 'Motor Cab', 'Motor Car', 'Maxi Cab', 'Ominibus', 
                                 'Omnibus For Private Use', 'Quadracycle Nontransport',
                                 'Quadracycle Transport', 'Ambulance'],
                  'Goods_n_Others' : ['Articulated Vehicles', 'Chassis Transport',
                                       'Fork Lift', 'Goods Carriage', 'Motor Grader',
                                       'Tractor For Commercial Use', 'Ecart',
                                       'Trailer For Commercial Use',
                                       'Three Wheeled Goods Vehicle',
                                       'Self Loading Concrete Mixer', 'Road Roller',
                                       'Loader', 'Crane For Private Use',
                                       'Vehicle Fitted With Construction Equipment']}

rta_DF['vehicleClass_'] =  rta_DF.vehicleClass.replace({value:
    key for key, values in vClass_mapping.items() for value in values})

In [33]:
columns = ['vehicleClass_', 'Category']
rta_DF[columns].drop_duplicates().sort_values(columns).reset_index(drop = True)

,vehicleClass_,Category
0,Agriculture,Non Transport
1,Auto_Rickshaw,Non Transport
2,Auto_Rickshaw,Transport
3,Goods_n_Others,Non Transport
4,Goods_n_Others,Transport
5,Motor_Car,Non Transport
6,Motor_Car,Transport
7,Motor_Cycle,Non Transport
8,Motor_Cycle,Transport


In [34]:
rta_DF.query(' SeatingCapacity == 0').vehicleClass_.value_counts()

vehicleClass_
Goods_n_Others    130438
Agriculture        96922
Motor_Cycle           60
Motor_Car             34
Name: count, dtype: Int64

#### 4.1.1 Correcting `Motor Cycle` data

In [35]:
pd.DataFrame(rta_DF.query('vehicleClass_ == "Motor_Cycle"').SeatingCapacity.value_counts() \
             .reset_index().sort_values('SeatingCapacity'))

,SeatingCapacity,count
3,0,60
1,1,5687
0,2,6766809
4,4,2
2,5,232


These values can't be true
- Seating Capacity 0 : Significant data issue, it’s illogical for motorcycles to have no seating.
- Seating Capacity 3 : Very few motorcycles with three seats.
- Seating Capacity 4 : Extremely rare, possibly indicating errors or mislabeling.
- Seating Capacity 5 : Possible misclassification.

In [36]:
cols_toShow = ['Model_Desc', 'Fuel', 'SeatingCapacity', 'Manufacturer_Name']
rta_DF.query('SeatingCapacity > 2 and \
              vehicleClass_ == "Motor_Cycle"')[cols_toShow].drop_duplicates()

,Model_Desc,Fuel,SeatingCapacity,Manufacturer_Name
1418439,PRAISE PRO BOV,Electric,5,M/S OKINAWA AUTOTECH PVT LTD
2480791,PULSAR 180 DTS-I BSVI,Petrol,5,BAJAJ AUTO LTD
3984943,TVS KING LS+FI BSVI,Hybrid,4,TVS MOTOR COMPANY LTD


In [37]:
print('>>> PRAISE PRO BOV')
display(rta_DF.query('Model_Desc == "PRAISE PRO BOV"').SeatingCapacity.value_counts())
print('\n>>> PULSAR 180 DTS-I BSVI')
display(rta_DF.query('Model_Desc == "PULSAR 180 DTS-I BSVI"').SeatingCapacity.value_counts())
print('\n>>> TVS KING LS+FI BSVI')
display(rta_DF.query('Model_Desc == "TVS KING LS+FI BSVI"').vehicleClass.value_counts())

>>> PRAISE PRO BOV


SeatingCapacity
2    2191
5      18
Name: count, dtype: int64


>>> PULSAR 180 DTS-I BSVI


SeatingCapacity
2    949
5    214
Name: count, dtype: int64


>>> TVS KING LS+FI BSVI


vehicleClass
Auto Rickshaw    74
Motor Cycle       2
Name: count, dtype: Int64

In [38]:
# https://www.bikewale.com/okinawa-bikes/praise/
rta_DF.loc[rta_DF.Model_Desc == 'PRAISE PRO BOV', 'SeatingCapacity'] = 2
# https://www.bikewale.com/bajaj-bikes/pulsar-180/
rta_DF.loc[rta_DF.Model_Desc == 'PULSAR 180 DTS-I BSVI', 'SeatingCapacity'] = 2
# https://trucks.tractorjunction.com/en/tvs-truck/deluxe-ls-fi-4s
rta_DF.loc[rta_DF.Model_Desc == 'TVS KING LS+FI BSVI', 'vehicleClass'] = 'Auto Rickshaw'

In [39]:
rta_DF.query('SeatingCapacity == 0 and \
              vehicleClass_ == "Motor_Cycle"')[cols_toShow].drop_duplicates()

,Model_Desc,Fuel,SeatingCapacity,Manufacturer_Name
957898,BMW R1200 R BSIV,Petrol,0,BMW INDIA PVT LTD
5293475,GLAMOUR BLACK & WHITE BSVI,Petrol,0,HERO MOTOCORP LTD
5382658,SCRAMBLER-DESERT SLED BSVI,Petrol,0,DUCATI INDIA PRIVATE LIMITED
5670864,YEZDI ROADSTER BSVI,Petrol,0,M/S. CLASSIC LEGENDS PVT LTD.
9512355,C 12I - MAX BOV,Electric,0,BGAUSS AUTO PVT LTD


In [40]:
first = True
for idx, value in rta_DF.query('SeatingCapacity == 0 and \
                    vehicleClass_ == "Motor_Cycle"')[cols_toShow].drop_duplicates().iterrows():
    if first:
        first = False
        print(f'>>> {value.iloc[0]}')
    else: print(f'\n>>> {value.iloc[0]}')
    display(rta_DF.query('Model_Desc == @value.iloc[0]').SeatingCapacity.value_counts())

>>> BMW R1200 R BSIV


SeatingCapacity
0    3
Name: count, dtype: int64


>>> GLAMOUR BLACK & WHITE BSVI


SeatingCapacity
0    18
Name: count, dtype: int64


>>> SCRAMBLER-DESERT SLED BSVI


SeatingCapacity
0    2
Name: count, dtype: int64


>>> YEZDI ROADSTER BSVI


SeatingCapacity
2    1752
0      36
Name: count, dtype: int64


>>> C 12I - MAX BOV


SeatingCapacity
0    1
Name: count, dtype: int64

In [41]:
# https://www.bikewale.com/bmw-bikes/r1200r/
rta_DF.loc[rta_DF.Model_Desc == 'BMW R1200 R BSIV', 'SeatingCapacity'] = 2
# https://www.bikewale.com/hero-bikes/glamour/
rta_DF.loc[rta_DF.Model_Desc == 'GLAMOUR BLACK & WHITE BSVI', 'SeatingCapacity'] = 2
# https://ducati.com.my/bikes/scrambler/scrambler-desert-sled
rta_DF.loc[rta_DF.Model_Desc == 'SCRAMBLER-DESERT SLED BSVI', 'SeatingCapacity'] = 2
# https://www.bikewale.com/yezdi-bikes/roadster/
rta_DF.loc[rta_DF.Model_Desc == 'YEZDI ROADSTER BSVI', 'SeatingCapacity'] = 2
# https://www.bikedekho.com/bgauss/c12i-max
rta_DF.loc[rta_DF.Model_Desc == 'C 12I - MAX BOV', 'SeatingCapacity'] = 2

In [42]:
# SeatingCapacity == 1
# https://www.bikewale.com/honda-bikes/dream/
rta_DF.loc[rta_DF.Model_Desc == 'CD 110 DREAM STD BSVI', 'SeatingCapacity'] = 2
# https://www.bikewale.com/tvs-bikes/apache-rr-310/
rta_DF.loc[rta_DF.Model_Desc == 'APACHE RR 310 BSVI', 'SeatingCapacity'] = 2
# https://www.bikewale.com/tvs-bikes/star-city/
rta_DF.loc[rta_DF.Model_Desc == 'TVS STAR CITY+ BSVI', 'SeatingCapacity'] = 2

# https://trucks.cardekho.com/en/trucks/bajaj/compact-4s/3-seater2000lpg
rta_DF.loc[rta_DF.Model_Desc == 'RE COMPACT LPG THL BSIV', 'SeatingCapacity'] = 4

#### 4.1.2 Correcting `Motor Car` data

In [43]:
first = True
for idx, value in rta_DF.query('SeatingCapacity == 0 and \
                    vehicleClass == "Motor Car"')[cols_toShow].drop_duplicates().iterrows():
    if first:
        first = False
        print(f'>>> {value.iloc[0]}')
    else: print(f'\n>>> {value.iloc[0]}')
    display(rta_DF.query('Model_Desc == @value.iloc[0]').SeatingCapacity.value_counts())

>>> CRETA 1.6 VTVT SX (O) EXECUTIVE BSIV


SeatingCapacity
5    146
0     25
Name: count, dtype: int64


>>> BMW 320D LUXURY LINE WITH AT BSIV


SeatingCapacity
5    12
0     9
Name: count, dtype: int64

In [44]:
# https://www.carwale.com/hyundai-cars/creta-2019-2020/sx-16-o-executive-petrol/
rta_DF.loc[rta_DF.Model_Desc == 'CRETA 1.6 VTVT SX (O) EXECUTIVE BSIV', 'SeatingCapacity'] = 5
# https://www.carwale.com/bmw-cars/3-series/320d-luxury-line/
rta_DF.loc[rta_DF.Model_Desc == 'BMW 320D LUXURY LINE WITH AT BSIV', 'SeatingCapacity'] = 5

#### 4.1.3 Correcting `Three Wheeled Goods Vehicle` data

In [45]:
first = True
for idx, value in rta_DF.query('SeatingCapacity == 4 and vehicleClass == \
                "Three Wheeled Goods Vehicle"')[cols_toShow].drop_duplicates().iterrows():
    if first:
        first = False
        print(f'>>> {value.iloc[0]}')
    else: print(f'\n>>> {value.iloc[0]}')
    display(rta_DF.query('Model_Desc == @value.iloc[0]').vehicleClass.value_counts())

>>> MAHINDRA ALFA PAX DX DIESEL BSIV


vehicleClass
Auto Rickshaw                  3249
Three Wheeled Goods Vehicle      12
Name: count, dtype: Int64


>>> RE MAXIMA DIESEL BSIV


vehicleClass
Auto Rickshaw                  529
Three Wheeled Goods Vehicle      4
Name: count, dtype: Int64


>>> RE COMPACT DIESEL BSIV


vehicleClass
Auto Rickshaw                  40367
Three Wheeled Goods Vehicle        4
Name: count, dtype: Int64

In [46]:
# https://mahindralastmilemobility.com/alfa-diesel-passenger
rta_DF.loc[rta_DF.Model_Desc == 'MAHINDRA ALFA PAX DX DIESEL BSIV', 'vehicleClass'] = \
                                                                           'Auto Rickshaw'
# https://trucks.cardekho.com/en/trucks/bajaj/compact-4s/3-seaterdiesel
rta_DF.loc[rta_DF.Model_Desc == 'RE MAXIMA DIESEL BSIV', 'vehicleClass'] = 'Auto Rickshaw'
# https://trucks.cardekho.com/en/trucks/bajaj/compact-4s/3-seaterdiesel
rta_DF.loc[rta_DF.Model_Desc == 'RE COMPACT DIESEL BSIV', 'vehicleClass'] = 'Auto Rickshaw'

#### 4.1.4 Correcting `Tractor For Agricultural Purpose` data

In [47]:
first = True
for idx, value in rta_DF.query('vehicleClass == "Tractor For Agricultural Purpose" and \
                                SeatingCapacity == 5')[cols_toShow].drop_duplicates().iterrows():
    if first:
        first = False
        print(f'>>> {value.iloc[0]}')
    else: print(f'\n>>> {value.iloc[0]}')
    display(rta_DF.query('Model_Desc == @value.iloc[0]').SeatingCapacity.value_counts())

>>> VST- SHAKTI -9045 DI VIRAAJ XT BSIIIA


SeatingCapacity
5    59
Name: count, dtype: int64

In [48]:
# https://tractorgyan.com/tractor/vst-shakti-viraaj-xp-9054-di/493
rta_DF.loc[rta_DF.Model_Desc == 'VST- SHAKTI -9045 DI VIRAAJ XT BSIIIA', 'SeatingCapacity'] = 1

#### 4.1.5 Correcting `Tractor For Agricultural Purpose` data

In [49]:
first = True
for idx, value in rta_DF.query('vehicleClass == "Tractor For Agricultural Purpose" and \
                                SeatingCapacity == 0')[cols_toShow].drop_duplicates().iterrows():
    if first:
        first = False
        print(f'>>> {value.iloc[0]}')
    else: print(f'\n>>> {value.iloc[0]}')
    display(rta_DF.query('Model_Desc == @value.iloc[0]').SeatingCapacity.value_counts())

>>> DASMESH-912 COMBINE HARVESTER


SeatingCapacity
1    4
0    2
Name: count, dtype: int64

In [50]:
# https://www.tractorjunction.com/harvester/9/dashmesh-912-combine-harvester/
rta_DF.loc[rta_DF.Model_Desc == 'DASMESH-912 COMBINE HARVESTER', 'SeatingCapacity'] = 1

In [51]:
print(f'>>> U-3518 TT BSIV')
display(rta_DF.query('Model_Desc == "U-3518 TT BSIV"').vehicleClass.value_counts())
print(f'\n>>> U-3518 TT BSIV')
display(rta_DF.query('Model_Desc == "U-3518 TT BSIV"').SeatingCapacity.value_counts())

>>> U-3518 TT BSIV


vehicleClass
Tractor For Commercial Use    17
Articulated Vehicles           2
Name: count, dtype: Int64


>>> U-3518 TT BSIV


SeatingCapacity
2    17
1     2
Name: count, dtype: int64

In [52]:
# https://trucks.cardekho.com/en/trucks/ashok-leyland/u-3518
rta_DF.loc[rta_DF.Model_Desc == 'U-3518 TT BSIV', 'SeatingCapacity'] = 2

#### 4.1.6 Correcting `Trailer For Commercial Use` data

In [53]:
first = True
for idx, value in rta_DF.query('vehicleClass == "Trailer For Commercial Use" and \
                                SeatingCapacity == 1')[cols_toShow].drop_duplicates().iterrows():
    if first:
        first = False
        print(f'>>> {value.iloc[0]}')
    else: print(f'\n>>> {value.iloc[0]}')
    display(rta_DF.query('Model_Desc == @value.iloc[0]').SeatingCapacity.value_counts())

>>> 5 TONS 2 WHEELER SEMI TIPPING TRAILER BSIII


SeatingCapacity
1    188
Name: count, dtype: int64


>>> 3 TONS 2 WHEELER WATER TANKER T IRON BODY


SeatingCapacity
0    910
1    113
Name: count, dtype: int64

In [54]:
rta_DF.loc[rta_DF.Model_Desc == '5 TONS 2 WHEELER SEMI TIPPING TRAILER BSIII',
                                                                        'SeatingCapacity'] = 0
rta_DF.loc[rta_DF.Model_Desc == '3 TONS 2 WHEELER WATER TANKER T IRON BODY',
                                                                        'SeatingCapacity'] = 0

In [55]:
display(rta_DF.query('Model_Desc == "JOHN DEERE 5405 4WD GEARPRO BSIV"')\
                    [['Model_Desc', 'vehicleClass', 'Fuel', 'SeatingCapacity']]\
                    .drop_duplicates().reset_index(drop = True))
rta_DF.loc[rta_DF.Model_Desc == 'JOHN DEERE 5405 4WD GEARPRO BSIV', 'Fuel'] = 'Diesel'
rta_DF.loc[rta_DF.Model_Desc == 'JOHN DEERE 5405 4WD GEARPRO BSIV', 'SeatingCapacity'] = 1

,Model_Desc,vehicleClass,Fuel,SeatingCapacity
0,JOHN DEERE 5405 4WD GEARPRO BSIV,Tractor For Agricultural Purpose,Petrol,2
1,JOHN DEERE 5405 4WD GEARPRO BSIV,Tractor For Agricultural Purpose,Diesel,1
2,JOHN DEERE 5405 4WD GEARPRO BSIV,Tractor For Commercial Use,Diesel,1
3,JOHN DEERE 5405 4WD GEARPRO BSIV,Tractor Driven Combined Harvester,Diesel,1


#### 4.1.7 Correcting `SeatingCapacity == 0` data

In [56]:
rta_DF.query('SeatingCapacity == 0 and \
              ~Model_Desc.str.contains("TRAILER|TRIALER", case = False) and\
              ~Model_Desc.str.contains("2 TONS|3TONS|3 TONS", case = False) and\
              ~Model_Desc.str.contains("5TONS|5TON|5 TON|5 TONS|5 TONNS", case = False) and\
              ~Model_Desc.str.contains("10TONS|10 TONS", case = False)',
              engine = 'python')[['Model_Desc', 'vehicleClass', 'SeatingCapacity']]\
              .drop_duplicates()

,Model_Desc,vehicleClass,SeatingCapacity
236389,APE XTRA DX DV BSIV,Three Wheeled Goods Vehicle,0
436667,TATA LPT3118/60CR 8X2 TC COWL 400LTS BSIV,Goods Carriage,0
553932,DASMESH-912COMIBNED HARVESTER,Tractor Driven Combined Harvester,0
2360293,JOHN DEERE 5405 4WD V14 TDCH DASMESH9124X4 BSIIIA,Tractor Driven Combined Harvester,0
6819896,SIMRAN 676 SELF PROPELLED COMBINEHARVESTER BSIII,Self Propelled Harvester,0


In [57]:
# https://trucks.cardekho.com/en/trucks/piaggio/ape/xtra-dv-diesel-bs-iv
rta_DF.loc[rta_DF.Model_Desc == 'APE XTRA DX DV BSIV', 'SeatingCapacity'] = 1
# https://trucks.cardekho.com/en/trucks/tata/lpt-3118-cowl
rta_DF.loc[rta_DF.Model_Desc == 'TATA LPT3118/60CR 8X2 TC COWL 400LTS BSIV',
                                                                        'SeatingCapacity'] = 2
rta_DF.loc[rta_DF.Model_Desc == 'DASMESH-912COMIBNED HARVESTER', 'SeatingCapacity'] = 1
rta_DF.loc[rta_DF.Model_Desc == 'JOHN DEERE 5405 4WD V14 TDCH DASMESH9124X4 BSIIIA',
                                                                        'SeatingCapacity'] = 1
rta_DF.loc[rta_DF.Model_Desc == 'SIMRAN 676 SELF PROPELLED COMBINEHARVESTER BSIII',
                                                                        'SeatingCapacity'] = 1

#### 4.2 Data Encoding

In [58]:
rta_DF['vehicleClass'] =  rta_DF.vehicleClass.replace({value:
    key for key, values in vClass_mapping.items() for value in values})

In [59]:
fuel_mapping = {'Others' : ['Hybrid', 'LPG/CNG']}
rta_DF.Fuel =  rta_DF.Fuel.replace({value:
                                key for key, values in fuel_mapping.items() for value in values})

In [60]:
fuel_encoded_ = pd.get_dummies(rta_DF.Fuel, prefix = 'fuel')
vehicleClass_encoded_ = pd.get_dummies(rta_DF.vehicleClass, prefix = 'vClass')

In [61]:
rta_DF['Brand_new_Vehicle'] = np.where(rta_DF.SecondVehicle == 'Y', 1, 0)
rta_DF['Pre_owned_Vehicle'] = np.where(rta_DF.SecondVehicle == 'N', 1, 0)
rta_DF['category_Transport'] = np.where(rta_DF.Category != 'Non Transport', 1, 0)
rta_DF['category_Non_Transport'] = np.where(rta_DF.Category == 'Non Transport', 1, 0)

In [62]:
for value in np.linspace(0.01, 0.96, 5):
    percentile_value = rta_DF['SeatingCapacity'].quantile(round(value, 3))
    print(f"{round(value, 3)*100}th Percentile: {percentile_value}")
for value in np.linspace(0.96, 1, 7):
    if value == 0.96: continue
    percentile_value = rta_DF['SeatingCapacity'].quantile(round(value, 3))
    print(f"{round(value, 3)*100}th Percentile: {percentile_value}")

1.0th Percentile: 0.0
24.8th Percentile: 2.0
48.5th Percentile: 2.0
72.2th Percentile: 2.0
96.0th Percentile: 5.0
96.7th Percentile: 5.0
97.3th Percentile: 6.0
98.0th Percentile: 7.0
98.7th Percentile: 7.0
99.3th Percentile: 7.0
100.0th Percentile: 66.0


In [63]:
rta_DF['seatCapacity_0'] = np.where(rta_DF.SeatingCapacity == 0, 1, 0)
rta_DF['seatCapacity_1_to_3'] = np.where((0 < rta_DF.SeatingCapacity) & \
                                                        (rta_DF.SeatingCapacity <= 3), 1, 0)
rta_DF['seatCapacity_4_to_6'] = np.where((3 < rta_DF.SeatingCapacity) & \
                                                        (rta_DF.SeatingCapacity <= 6), 1, 0)
rta_DF['seatCapacity_above_6'] = np.where(rta_DF.SeatingCapacity > 6, 1, 0)

In [64]:
print(rta_DF.shape)
rta_DF.head()

(9608322, 20)


,District,Model_Desc,vehicleClass,Fuel,SeatingCapacity,fromdate,TempRegnNo,Category,SecondVehicle,makeYear,Manufacturer_Name,vehicleClass_,Brand_new_Vehicle,Pre_owned_Vehicle,category_Transport,category_Non_Transport,seatCapacity_0,seatCapacity_1_to_3,seatCapacity_4_to_6,seatCapacity_above_6
0,Adilabad,ACTIVA 5GWEAS&KS&DCBS(CBS)WSEHEETWHEEL BSIV,Motor_Cycle,Petrol,2,2019-01-01,TS01LTR7367,Non Transport,N,2018-01-11,HONDA MOTORCYCLE&SCOOTER(I)P L,Motor_Cycle,0,1,0,1,0,1,0,0
1,Adilabad,MAHINDRA XUV5OO FWD W11 BSIV,Motor_Car,Diesel,7,2019-01-01,TS16XTR5445,Non Transport,Y,2018-01-08,M/S MAHINDRA &MAHINDRA LTD,Motor_Car,1,0,0,1,0,0,0,1
2,Adilabad,LIVO WK S &S S W F D &R D B W ALLOYWHEELS BSIV,Motor_Cycle,Petrol,2,2019-01-01,TS01LTR7329,Non Transport,N,2018-01-06,HONDA MOTORCYCLE&SCOOTER(I)P L,Motor_Cycle,0,1,0,1,0,1,0,0
3,Adilabad,"""PASSION PRO""(I3S-SELF-DRUM-CAST) BSIV",Motor_Cycle,Petrol,2,2019-01-01,TS01LTR7424,Non Transport,N,2018-01-12,HERO MOTOCORP LTD,Motor_Cycle,0,1,0,1,0,1,0,0
4,Adilabad,CB SHINE W F DRUM B K S&E S W CAST WHEELS BSIV,Motor_Cycle,Petrol,2,2019-01-01,TS01LTR7374,Non Transport,N,2018-01-11,HONDA MOTORCYCLE&SCOOTER(I)P L,Motor_Cycle,0,1,0,1,0,1,0,0


In [65]:
dropCols = ['TempRegnNo', 'Manufacturer_Name', 'Model_Desc', 'makeYear', 'SeatingCapacity',
            'vehicleClass_', 'Fuel', 'vehicleClass', 'Category', 'SecondVehicle']
rta_DF = rta_DF.drop(dropCols, axis = 1)
rta_DF = pd.concat([rta_DF, vehicleClass_encoded_, fuel_encoded_], axis = 1)

In [66]:
current_size = rta_DF.shape
missingCount = sum(rta_DF.isnull().sum())

print(f'Shape of initial RTA Registration Dataset  : {original_size}')
print(f'Shape of present RTA Registration Dataset  : {current_size}')
print(f'No.of issing values in the present dataset : {missingCount}\n')

rta_DF.head()

Shape of initial RTA Registration Dataset  : (9608322, 11)
Shape of present RTA Registration Dataset  : (9608322, 20)
No.of issing values in the present dataset : 0



,District,fromdate,Brand_new_Vehicle,Pre_owned_Vehicle,category_Transport,category_Non_Transport,seatCapacity_0,seatCapacity_1_to_3,seatCapacity_4_to_6,seatCapacity_above_6,vClass_Agriculture,vClass_Auto_Rickshaw,vClass_Goods_n_Others,vClass_Motor_Car,vClass_Motor_Cycle,fuel_Diesel,fuel_Electric,fuel_Non_Fuel,fuel_Others,fuel_Petrol
0,Adilabad,2019-01-01,0,1,0,1,0,1,0,0,False,False,False,False,True,False,False,False,False,True
1,Adilabad,2019-01-01,1,0,0,1,0,0,0,1,False,False,False,True,False,True,False,False,False,False
2,Adilabad,2019-01-01,0,1,0,1,0,1,0,0,False,False,False,False,True,False,False,False,False,True
3,Adilabad,2019-01-01,0,1,0,1,0,1,0,0,False,False,False,False,True,False,False,False,False,True
4,Adilabad,2019-01-01,0,1,0,1,0,1,0,0,False,False,False,False,True,False,False,False,False,True


#### 4.3 Data Groupping

Now we have day-to-day values, for better comparision and analysis, instead of day-to-day values, converting into Month wise data.

In [67]:
rta_DF['Year'] = rta_DF.fromdate.dt.year
rta_DF['Month'] = rta_DF.fromdate.dt.month

rta_Grouped = rta_DF.drop('fromdate', axis = 1).groupby(
                            ['District', 'Year', 'Month']).agg('sum').reset_index()

**NOTE :**

Creating new `Date` column, `Day` will be 01 for all (it's monthly total count,ie summation, so taking *Day* as 01), Month and Year will come from `rta_Grouped` dataframes `Month` and `Year` columns respectively.

In [68]:
rta_Grouped['Month'] = pd.to_datetime(rta_Grouped.apply(lambda row :
                            f"01/{row.Month}/{row.Year}", axis = 1), format = '%d/%m/%Y')

In [69]:
# Removing unwanted columns
# rta_Grouped.drop(['Year', 'Month'], axis = 1, inplace = True)
rta_Grouped.drop('Year', axis = 1, inplace = True)

# Ordering column names
col_order = ['Month', 'District', 'vClass_Agriculture', 'vClass_Motor_Cycle',
             'vClass_Auto_Rickshaw', 'vClass_Motor_Car', 'vClass_Goods_n_Others',
             'seatCapacity_0', 'seatCapacity_1_to_3', 'seatCapacity_4_to_6',
             'seatCapacity_above_6', 'fuel_Petrol', 'fuel_Diesel', 'fuel_Electric',
             'fuel_Others', 'fuel_Non_Fuel', 'Brand_new_Vehicle', 'Pre_owned_Vehicle',
             'category_Transport', 'category_Non_Transport']
rta_Grouped = rta_Grouped[col_order]

# Sorting dataframe
rta_Grouped = rta_Grouped.sort_values(['Month', 'District']).reset_index(drop = True)

current_size = rta_DF.shape
grouped_size = rta_Grouped.shape
missingCount = sum(rta_Grouped.isnull().sum())
print(f'Shape of initial RTA Registration Dataset\t  : {original_size}')
print(f'Shape of present grouped RTA Registration Dataset : {grouped_size}')
print(f'No.of issing values in the present dataset\t  : {missingCount}\n')

rta_Grouped.head()

Shape of initial RTA Registration Dataset	  : (9608322, 11)
Shape of present grouped RTA Registration Dataset : (2257, 20)
No.of issing values in the present dataset	  : 0



,Month,District,vClass_Agriculture,vClass_Motor_Cycle,vClass_Auto_Rickshaw,vClass_Motor_Car,vClass_Goods_n_Others,seatCapacity_0,seatCapacity_1_to_3,seatCapacity_4_to_6,seatCapacity_above_6,fuel_Petrol,fuel_Diesel,fuel_Electric,fuel_Others,fuel_Non_Fuel,Brand_new_Vehicle,Pre_owned_Vehicle,category_Transport,category_Non_Transport
0,2019-01-01,Adilabad,34,1322,100,113,211,26,1535,185,34,1365,379,2,8,26,29,1751,311,1469
1,2019-01-01,Bhadradri Kothagudem,190,4181,319,251,380,156,4590,533,42,4287,872,0,6,156,106,5215,715,4606
2,2019-01-01,Hyderabad,0,21343,804,4086,1060,4,22385,4250,654,23341,2969,96,883,4,1838,25455,3272,24021
3,2019-01-01,Jagtial,146,4007,65,165,147,108,4192,209,21,4081,339,0,2,108,209,4321,215,4315
4,2019-01-01,Jangaon,74,2265,62,87,194,52,2481,129,20,2298,330,0,2,52,74,2608,256,2426


In [70]:
print('>>> Descriptive statistics of grouped RTA Registration Data\n')
rta_Grouped.describe()

>>> Descriptive statistics of grouped RTA Registration Data



,Month,vClass_Agriculture,vClass_Motor_Cycle,vClass_Auto_Rickshaw,vClass_Motor_Car,vClass_Goods_n_Others,seatCapacity_0,seatCapacity_1_to_3,seatCapacity_4_to_6,seatCapacity_above_6,fuel_Petrol,fuel_Diesel,fuel_Electric,fuel_Others,fuel_Non_Fuel,Brand_new_Vehicle,Pre_owned_Vehicle,category_Transport,category_Non_Transport
count,2257,2257.0,2257.0,2257.0,2257.0,2257.0,2257.000000,2257.000000,2257.000000,2257.000000,2257.0,2257.0,2257.0,2257.0,2257.0,2257.000000,2257.000000,2257.000000,2257.000000
mean,2022-01-08 16:48:03.828090624,153.879043,3000.792202,103.157288,709.360656,289.931768,100.766061,3340.904298,706.244572,109.206026,3400.225078,567.797519,105.459016,83.372619,100.266726,309.977847,3947.143110,454.093930,3803.027027
min,2019-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.000000,6.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,8.000000,0.000000,6.000000
25%,2020-07-01 00:00:00,52.0,965.0,19.0,121.0,98.0,34.000000,1120.000000,141.000000,18.000000,1034.0,195.0,2.0,2.0,34.0,59.000000,1297.000000,156.000000,1209.000000
50%,2022-02-01 00:00:00,110.0,1627.0,43.0,224.0,190.0,69.000000,1910.000000,255.000000,35.000000,1758.0,355.0,18.0,15.0,68.0,103.000000,2231.000000,283.000000,2008.000000
75%,2023-07-01 00:00:00,214.0,2831.0,104.0,430.0,353.0,125.000000,3271.000000,529.000000,68.000000,3086.0,642.0,55.0,67.0,124.0,175.000000,3836.000000,513.000000,3514.000000
max,2024-12-01 00:00:00,1099.0,35432.0,2290.0,9014.0,2151.0,1087.000000,36692.000000,7922.000000,2000.000000,39703.0,5268.0,2954.0,2374.0,1087.0,4537.000000,42089.000000,5956.000000,42076.000000
std,NaN,144.990777,4192.39069,196.363039,1466.653814,300.406938,108.712628,4420.044908,1317.182360,228.854573,4950.592644,655.1734,318.503173,211.219861,108.342659,646.148294,5271.811591,535.988407,5412.540419


#### 4.4 Saving Processed Registration Dataset

In [71]:
# Saving DataFrame to Parquet

rta_path = f'{processed_file_path}/RTA_Reg_Data_Processed.csv'
print(f'Saving processed RTA Registration Dataset to "{rta_path}"')
if not os.path.isfile(rta_path):
    rta_Grouped.to_csv(rta_path, index = False)

Saving processed RTA Registration Dataset to "Data/processed/RTA_Reg_Data_Processed.csv"
